# Carteira B3 V2 — descoberta automática, segmentos e seleção

Esta versão **não usa uma lista fixa de nove ações**. A cada execução descobre o catálogo de ações/units fornecido pela API pública brapi, confere todas as páginas e exclui fundos, ETFs, BDRs e direitos/recibos de subscrição. O universo não fica limitado ao Ibovespa.

**Fluxo:** catálogo → setores/segmentos escolhidos → histórico de todos os ativos do escopo → cobertura, liquidez e preço → ranking e seleção automática → otimização de pesos → compras inteiras e caixa.

A classificação e a cobertura são do provedor, não uma certificação da lista oficial integral da B3. O notebook mostra contagens e registra exclusões/falhas. Sem dados suficientes, não inventa preços. A consulta de catálogo não exige token; os históricos usam Yahoo Finance/yfinance.

As regras de regressão contra Ibovespa e covariância da planilha foram preservadas com correções estatísticas. Os novos critérios de seleção são **premissas editáveis**, pois a planilha não define como selecionar ações de toda a bolsa.


In [ ]:
%pip -q install "numpy>=1.26,<3" "pandas>=2.2,<3" "scipy>=1.13,<2" "yfinance>=0.2.54,<2" "ipywidgets>=8,<9" "matplotlib>=3.8,<4"


In [ ]:
from google.colab import drive,output
drive.mount('/content/drive')
output.enable_custom_widget_manager()
from pathlib import Path
PASTA_DRIVE=Path('/content/drive/MyDrive/Carteira_Ativos') # reutiliza histórico da primeira versão
PASTA_DRIVE.mkdir(parents=True,exist_ok=True)
print('Pasta de dados:',PASTA_DRIVE)


In [ ]:
# Carrega o motor versionado no repositório.
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    raise RuntimeError('Abra o notebook a partir da raiz do repositório clonado.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.engine import *
from src.universe_b3 import *
print('Motor B3 versionado carregado')


## 1. Descobrir ações e segmentos

Execute esta célula para atualizar a lista completa do provedor. O CSV `universo_b3.csv` é atualizado por inteiro, pois empresas, tickers e classificações podem mudar. **O histórico de preços permanece incremental.** O programa confere quantidade declarada, paginação, tipos e duplicatas antes de salvar a lista.

Se a API estiver indisponível e você quiser conscientemente usar o último catálogo salvo, altere `USAR_CATALOGO_SALVO=True`. A data do catálogo antigo será exibida. Não há retorno silencioso à lista de nove ações.


In [ ]:
import pandas as pd,numpy as np,json
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display,clear_output
USAR_CATALOGO_SALVO=False
if USAR_CATALOGO_SALVO:
    universo=pd.read_csv(PASTA_DRIVE/'universo_b3.csv')
    meta=json.loads((PASTA_DRIVE/'catalogo_meta.json').read_text())
    print('ATENÇÃO: usando catálogo salvo, sem verificar novas listagens.')
else:
    universo,meta=refresh_catalog(PASTA_DRIVE)
print('Registros recebidos da fonte:',meta['returned_count'])
print('Ações/units para análise:',len(universo))
print('Tipos/códigos excluídos:',meta['excluded_other_types'])
print('Setores:',universo.sector_pt.nunique(),'| segmentos:',universo.segment.nunique())
print('Atualização do catálogo:',meta['fetched_at'])
display(universo[['ticker','name','sector_pt','segment']].head(20))


## 2. Configurar a busca e a carteira

Todos os setores/segmentos começam selecionados. Você pode restringir o escopo usando Ctrl/Cmd nos seletores. A busca histórica inclui **todos os ativos do escopo**, antes do ranking; não baixa apenas os integrantes do Ibovespa ou os dez escolhidos.

Padrões editáveis: liquidez média de R$ 1 milhão/dia em 60 pregões, 24 meses completos, até 10 ativos selecionados, máximo de 25% do capital por ativo e compra de uma ação por unidade. A seleção padrão prioriza liquidez e os pesos minimizam variância. Alternativas: ranking por Sharpe histórico ou por menor volatilidade; pesos por maior Sharpe estimado.

O máximo de ativos não é um mínimo de compras: alguns pesos/quantidades podem ser zero. O modo “uma classe por prefixo” evita ON/PN/units do mesmo prefixo; não resolve todas as relações societárias. Limites por segmento/setor podem exigir uma seleção diferente dos primeiros nomes do ranking.

**Primeira execução:** centenas de consultas podem levar vários minutos. O histórico é salvo após cada ativo; consultas validadas no mesmo dia são reutilizadas. Aumente a pausa se houver limites de acesso. Duas falhas consecutivas HTTP 429 interrompem novas consultas e bloqueiam o cálculo até a retomada.


In [ ]:
style={'description_width':'240px'}
layout=widgets.Layout(width='730px')
capital=widgets.FloatText(value=10000.,description='Valor disponível (R$)',style=style,layout=layout)
setores=widgets.SelectMultiple(options=sorted(universo.sector_pt.unique()),value=tuple(sorted(universo.sector_pt.unique())),description='Setores',style=style,layout=widgets.Layout(width='730px',height='170px'))
segmentos=widgets.SelectMultiple(options=sorted(universo.segment.unique()),value=tuple(sorted(universo.segment.unique())),description='Segmentos',style=style,layout=widgets.Layout(width='730px',height='210px'))
escopo_label=widgets.HTML()
def escopo_atual():
    return universo[universo.sector_pt.isin(setores.value)&universo.segment.isin(segmentos.value)].copy()
def atualizar_contagem(_=None):
    escopo_label.value=f'<b>{len(escopo_atual())} ações/units no escopo; todas serão consultadas antes da seleção.</b>'
def atualizar_segmentos(change):
    opts=sorted(universo.loc[universo.sector_pt.isin(setores.value),'segment'].unique())
    segmentos.options=opts;segmentos.value=tuple(opts);atualizar_contagem()
setores.observe(atualizar_segmentos,names='value')
segmentos.observe(atualizar_contagem,names='value');atualizar_contagem()
inicio=widgets.Text(value=str((pd.Timestamp.now(tz='America/Sao_Paulo')-pd.DateOffset(years=6)).date()),description='Início (AAAA-MM-DD)',style=style,layout=layout)
janela=widgets.IntSlider(value=24,min=12,max=60,description='Meses completos',style=style,layout=layout)
liquidez=widgets.FloatText(value=1000000.,description='Liquidez média mínima (R$/dia)',style=style,layout=layout)
max_ativos=widgets.IntSlider(value=10,min=1,max=30,description='Máximo de ativos selecionados',style=style,layout=layout)
ranking=widgets.Dropdown(options=[('Maior liquidez média','liquidity'),('Maior Sharpe histórico','sharpe'),('Menor volatilidade histórica','volatility')],value='liquidity',description='Seleção automática por',style=style,layout=layout)
objetivo=widgets.Dropdown(options=[('Menor variância','min_variance'),('Maior Sharpe estimado','max_sharpe')],description='Otimização dos pesos',style=style,layout=layout)
limite_ativo=widgets.FloatSlider(value=25,min=5,max=100,step=5,description='Máximo por ativo (%)',style=style,layout=layout)
limite_grupo=widgets.FloatSlider(value=100,min=5,max=100,step=5,description='Máximo por grupo (%)',style=style,layout=layout)
grupo=widgets.Dropdown(options=[('Segmento','segment'),('Setor','sector_pt')],description='Grupo para limite',style=style,layout=layout)
uma_classe=widgets.Checkbox(value=True,description='Manter só a classe mais líquida de cada prefixo',layout=layout)
parcial=widgets.Checkbox(value=True,description='Continuar com falhas isoladas, mostrando os ativos não avaliados',layout=layout)
rf=widgets.FloatText(value=10.,description='Premissa taxa livre anual (%)',style=style,layout=layout)
usar_cenario=widgets.Checkbox(value=False,description='Informar cenário mensal do Ibovespa',layout=layout)
cenario=widgets.FloatText(value=0.,description='Cenário mensal Ibovespa (%)',style=style,layout=layout)
lote=widgets.Dropdown(options=[('Uma ação — fracionário',1),('Lote de 100 ações',100)],description='Lote',style=style,layout=layout)
taxa=widgets.FloatText(value=0.,description='Custos variáveis (%)',style=style,layout=layout)
fixo=widgets.FloatText(value=0.,description='Reserva fixa por ativo (R$)',style=style,layout=layout)
pausa=widgets.FloatSlider(value=.4,min=0,max=5,step=.1,description='Pausa entre consultas (s)',style=style,layout=layout)
balancos=widgets.Checkbox(value=False,description='Consultar balanços anuais apenas dos ativos selecionados',layout=layout)
botao=widgets.Button(description='Buscar B3 e montar carteira',button_style='success',layout=widgets.Layout(width='330px'))
saida=widgets.Output()
display(widgets.VBox([capital,setores,segmentos,escopo_label,inicio,janela,liquidez,max_ativos,ranking,objetivo,limite_ativo,limite_grupo,grupo,uma_classe,parcial,rf,usar_cenario,cenario,lote,taxa,fixo,pausa,balancos,botao,saida]))


## 3. Habilitar o botão e executar

Execute esta célula e clique no botão acima. A taxa livre de risco é uma premissa sua, não uma consulta à Selic vigente. Sem cenário informado, usa a média mensal histórica do Ibovespa; nesse caso a previsão da regressão coincide com a média histórica da ação.

Seleção e pesos são etapas distintas: um MILP escolhe os nomes mais bem colocados no ranking que admitem os limites de peso; depois, o otimizador calcula pesos nesse conjunto. Isso **não prova a melhor combinação global** de ações por risco/retorno. As regras de ranking e liquidez não estavam na planilha original e estão expostas para alteração.


In [ ]:
def executar(_):
    botao.disabled=True
    with saida:
        clear_output(wait=True)
        try:
            u=escopo_atual()
            if u.empty:raise ValueError('Selecione pelo menos um setor e segmento')
            if capital.value<=0:raise ValueError('Informe capital positivo')
            if rf.value<0 or liquidez.value<0 or taxa.value<0 or fixo.value<0:raise ValueError('Premissas negativas inválidas')
            pd.Timestamp(inicio.value)
            now=pd.Timestamp.now(tz='America/Sao_Paulo').strftime('%Y%m%d_%H%M%S')
            destino=PASTA_DRIVE/'resultados_b3_v2'/now
            destino.mkdir(parents=True,exist_ok=True)
            cfg=dict(capital=capital.value,start=inicio.value,window=janela.value,min_liquidity=liquidez.value,
                max_assets=max_ativos.value,max_asset=limite_ativo.value/100,max_segment=limite_grupo.value/100,
                group_column=grupo.value,ranking=ranking.value,objective=objetivo.value,
                one_per_prefix=uma_classe.value,allow_partial=parcial.value,rf_annual=rf.value/100,
                market_scenario=cenario.value/100 if usar_cenario.value else None,
                lot=lote.value,fee_rate=taxa.value/100,fixed_fee=fixo.value,
                sectors=list(setores.value),segments=list(segmentos.value),scope_count=len(u),
                source_catalog=meta)
            (destino/'configuracao.json').write_text(json.dumps(cfg,ensure_ascii=False,indent=2),encoding='utf-8')
            save_csv(u,destino/'escopo_selecionado.csv')
            print(f'Consultando {len(u)} ações/units e Ibovespa. Checkpoints após cada ativo.')
            status=display('Iniciando consulta…',display_id=True)
            historico,logs,bloqueado=refresh_history(PASTA_DRIVE,u.ticker.tolist(),inicio.value,
                progress=lambda msg:status.update(msg),pause=pausa.value)
            save_csv(logs,destino/'diagnostico_coleta.csv')
            falhas=logs[logs.status.ne('ok')]
            print('Atualizações OK/cache:',int(logs.status.eq('ok').sum()),'| Falhas/não consultados:',len(falhas))
            if not falhas.empty:display(falhas[['ticker','status','message']].head(30))
            if bloqueado:raise RuntimeError('HTTP 429 repetido: consultas pausadas. Histórico preservado; aguarde e clique novamente para retomar. Nenhuma carteira foi calculada nesta execução.')
            resultado=run_b3(historico,u,logs,capital.value,window=janela.value,min_liquidity=liquidez.value,
                max_assets=max_ativos.value,max_asset=cfg['max_asset'],max_segment=cfg['max_segment'],
                ranking=ranking.value,one_per_prefix=uma_classe.value,allow_partial=parcial.value,
                objective=objetivo.value,rf_annual=cfg['rf_annual'],market_scenario=cfg['market_scenario'],
                lot=lote.value,fee_rate=cfg['fee_rate'],fixed_fee=fixo.value,
                group_column=grupo.value,diagnostic_folder=destino)
            save_result(resultado,destino)
            save_csv(resultado['selected'],destino/'ativos_selecionados.csv')
            s=resultado['summary']
            print('Escopo:',s['universe_considered'],'| Elegíveis:',s['eligible_count'],'| Selecionados:',s['selected_count'],'| Com compra:',s['bought_count'])
            if s['download_failed_count']:print('ATENÇÃO: carteira calculada com cobertura parcial;',s['download_failed_count'],'ativos não avaliados por falha de consulta. Veja diagnóstico e triagem.')
            print(f"Capital: R$ {s['capital']:,.2f} | Investido: R$ {s['invested']:,.2f} | Caixa: R$ {s['cash']:,.2f}")
            print(f"Cotações: {s['quote_date']} | Meses: {s['first_month']} a {s['last_month']}")
            print(f"Pesos teóricos: retorno mensal estimado {100*s['expected_month']:.2f}% | volatilidade mensal {100*s['volatility_month']:.2f}%")
            print(f"Carteira arredondada, apenas ações: retorno mensal {100*s['rounded_risky_expected_month']:.2f}% | volatilidade {100*s['rounded_risky_volatility_month']:.2f}%")
            display(resultado['plan'].round(6))
            print('Melhores posições do ranking (antes dos pesos):')
            display(resultado['eligible'].head(20))
            excluidos=resultado['screening'][~resultado['screening'].eligible]
            print('Excluídos da elegibilidade:',len(excluidos),'— lista completa em triagem.csv')
            display(excluidos[['ticker','exclusion']].head(20))
            fig,ax=plt.subplots(1,2,figsize=(13,4))
            mc=resultado['monte_carlo']
            if not mc.empty:
                dots=ax[0].scatter(mc.volatility_month*100,mc.expected_month*100,c=mc.sharpe_month,s=5,alpha=.4)
                fig.colorbar(dots,ax=ax[0],label='Sharpe mensal')
            ax[0].scatter(s['volatility_month']*100,s['expected_month']*100,c='red',marker='*',s=90)
            ax[0].set(xlabel='Volatilidade mensal (%)',ylabel='Retorno estimado mensal (%)',title='Monte Carlo dos ativos selecionados')
            resultado['plan'].set_index('ticker')[['target_weight','actual_weight_capital']].mul(100).plot.bar(ax=ax[1])
            ax[1].set(ylabel='% do capital',title='Pesos teóricos e compras');ax[1].legend(['Teórico','Efetivo'])
            fig.tight_layout();plt.show()
            if balancos.value:
                raw,erros=update_fundamentals(PASTA_DRIVE/'fundamentos.csv',resultado['selected'].ticker.tolist())
                manual=PASTA_DRIVE/'fundamentos_revisados.csv'
                if manual.exists():raw=apply_fundamental_overrides(raw,pd.read_csv(manual))
                ratios=fundamental_ratios(raw)
                if not ratios.empty:
                    ratios=ratios[ratios.ticker.isin(resultado['selected'].ticker)]
                    save_csv(ratios,destino/'indicadores_balanco.csv');display(ratios)
                if not erros.empty:save_csv(erros,destino/'falhas_balancos.csv');display(erros)
            import importlib.metadata as im
            (destino/'versoes.json').write_text(json.dumps({m:im.version(m) for m in ['numpy','pandas','scipy','yfinance']},indent=2))
            print('Arquivos salvos:',destino)
            print('Prova de ótimo do MILP de seleção por ranking:',s['selection_solver_optimal'])
            print('Prova de ótimo da compra inteira:',s['solver_optimal'])
            globals()['ULTIMO_RESULTADO_B3']=resultado
        except Exception as e:
            print('Execução não concluída:',str(e))
            print('Confira atualizacoes.csv e a pasta da execução. Os checkpoints do histórico foram preservados.')
        finally:botao.disabled=False
botao.on_click(executar)
print('Pronto. Configure os parâmetros e clique em Buscar B3 e montar carteira.')


## Arquivos e metodologia

Na pasta principal do Drive: `universo_b3.csv`, `catalogo_meta.json`, `historico.csv`, `atualizacoes.csv` e, opcionalmente, `fundamentos.csv`. Resultados em `resultados_b3_v2/data_hora/`: escopo, parâmetros, diagnóstico, triagem, ranking de **todos** os elegíveis, seleção, compras, regressões, covariância e Monte Carlo.

O histórico tem chave data/ativo; cada nova execução completa a série com sobreposição de 14 dias. Ajustes revisados, dividendos ou desdobramentos podem exigir reconsulta completa do ativo afetado. A cotação do dia corrente é excluída. Meses de análise são completos e alinhados pelo último pregão mensal do Ibovespa, sem preenchimento de preço faltante. Liquidez usa média de fechamento × volume dos últimos 60 pregões, com pelo menos 90% de cobertura diária; preço × volume é uma aproximação do giro financeiro.

Triagem exige janela completa, cotações na mesma data do benchmark, liquidez mínima e um lote que caiba no limite monetário por ativo. As exclusões distinguem falha de consulta de falta de cobertura ou liquidez. A seleção usa ranking e viabilidade dos limites; pesos usam covariância amostral. Monte Carlo sorteia pesos dos nomes selecionados, sem simular toda combinação de ações da bolsa.

Regressão: ação sobre índice, previsão `alfa+beta×cenário`. A taxa livre de risco de 10% é apenas um valor inicial editável. Ranking histórico e pesos calculados na mesma amostra não demonstram desempenho fora da amostra. Não há backtest nesta versão, nem envio de ordens.

Consulta opcional de balanços: ROE=LL/PL, ROA=EBIT/Ativo conforme planilha, GAF=ROE/ROA, GAO=variação EBIT/variação receita. Dados anuais podem faltar; não são filtros automáticos de seleção. Ki exato exige despesa financeira total e dívida onerosa revisadas, enquanto a proxy juros/dívida total é identificada separadamente. Não compare bancos e empresas industriais sem ajustar a metodologia.

Fontes: [catálogo público brapi e paginação](https://brapi.dev/docs/acoes/list), [históricos yfinance](https://ranaroussi.github.io/yfinance/reference/yfinance.price_history.html), [SciPy MILP](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.milp.html). Verifique condições de uso dos dados antes de publicar um produto comercial.

O ZIP desta versão contém o motor e um app Streamlit que também usa descoberta automática. O app mantém cache por sessão e oferece download/upload do histórico; persistência automática em Drive no servidor exige integração própria. Validação: catálogo real consultado e testes de cálculo/atualização executados com dados sintéticos; preços atuais não foram integralmente coletados neste ambiente.
